<a href="https://colab.research.google.com/github/AbdulrahmanB-25/Machine_Learning_Competition/blob/main/EDA_of_DataSets/EDA_Riyadh_Real_Estate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
import sqlite3
import pandas as pd
import os
import glob
from google.colab import files

# Display settings to see all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# --- 1. Download Dataset ---
print("Downloading database from Kaggle...")
dataset_path = kagglehub.dataset_download("mohdph/saudi-arabia-real-estate-dataset")

db_files = glob.glob(os.path.join(dataset_path, "*.db"))
if not db_files:
    raise FileNotFoundError("No .db file found. Check the Kaggle dataset structure.")

db_path = db_files[0]
print(f"Found database: {os.path.basename(db_path)}")

100%|██████████| 351M/351M [00:02<00:00, 133MB/s]

Extracting files...


Found database: database.db


In [ ]:
# --- 2. Connect and Extract Riyadh Data ---
conn = sqlite3.connect(db_path)

try:
    # Identify the table name automatically
    table_name = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn).iloc[0, 0]
    print(f"Table identified: {table_name}")

    # Load ONLY Riyadh data
    print("Extracting Riyadh records...")
    df_riyadh = pd.read_sql_query(f"SELECT * FROM {table_name} WHERE city = 'الرياض'", conn)
    print(f"Loaded {len(df_riyadh):,} records.")

finally:
    conn.close()

Table identified: Listings
Extracting Riyadh records...
Loaded 423,720 records.


In [ ]:
# --- 3. Feature Selection & Data Cleaning ---

# The 13 specific features identified for the model
selected_cols = [
    'price', 'area', 'category', 'beds', 'livings', 'wc',
    'age', 'ketchen', 'ac', 'furnished',
    'location.lat', 'location.lng', 'user.review'
]

# 1. Slice the dataframe to keep only the 13 raw features
df_spatial = df_riyadh[selected_cols].copy()

# 2. Data Cleaning
# Remove rows without coordinates to ensure they can be mapped
df_spatial = df_spatial.dropna(subset=['location.lat', 'location.lng'])

# Remove rows where price or area are zero to keep the data valid
df_spatial = df_spatial[(df_spatial['price'] > 0) & (df_spatial['area'] > 0)]

print(f"Data ready. Total records: {len(df_spatial):,}")
print(f"Current features: {list(df_spatial.columns)}")
df_spatial.head()

Data ready. Total records: 423,519
Current features: ['price', 'area', 'category', 'beds', 'livings', 'wc', 'age', 'ketchen', 'ac', 'furnished', 'location.lat', 'location.lng', 'user.review']


,price,area,category,beds,livings,wc,age,ketchen,ac,furnished,location.lat,location.lng,user.review
0,1958400.0,816.0,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,24.548930,46.781390,3.83
1,15078000.0,1077.0,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,24.754130,46.724820,4.56
2,1050000.0,875.0,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,24.560830,46.795360,4.08
3,8750000.0,1250.0,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,24.644165,46.659703,4.53
4,53625000.0,10725.0,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,24.853706,46.669090,4.19



### **Part 4: Feature Engineering & Data Enrichment**

In this section, we transform the raw data into "Smart Features" that our ranking model can use to evaluate Riyadh's neighborhoods. We have performed four specific transformations:

1.  **Normalization of Coordinates & Schema:**
    * **Action:** Renamed `location.lat` to `lat`, `location.lng` to `lng`, and fixed the `ketchen` typo to `kitchen`.
    * **Reason:** This ensures compatibility with standard Geospatial libraries (like GeoPandas) and prevents naming errors in our Streamlit filters.

2.  **Value-per-Area Metric (`price_per_m2`):**
    * **Action:** Created by dividing `price` by `area`.
    * **Reason:** Total price is biased by the size of the property. Price per square meter is the industry standard for comparing real estate value across different districts fairly.

3.  **Property Quality Score (`quality_score`):**
    * **Action:** Aggregated three boolean features (`kitchen` + `ac` + `furnished`) into a single score ranging from 0 to 3.
    * **Reason:** This provides a quick "Ready-to-Move-In" metric. A higher score indicates a more premium or fully-equipped listing.

4.  **Categorical Mapping (`category_name`):**
    * **Action:** Mapped integer category IDs to their official Arabic/English descriptions (e.g., 3 → "Villa (Sell)").
    * **Reason:** This improves human readability and allows for intuitive dropdown filters in the final Streamlit application.



In [ ]:
# --- 4. Feature Engineering ---

# 1. Column Renaming & Typo Fixing
# Renaming coordinates for spatial tools and fixing the 'ketchen' typo
feature_mapping = {
    'location.lat': 'lat',
    'location.lng': 'lng',
    'ketchen': 'kitchen'
}
df_spatial = df_spatial.rename(columns=feature_mapping)

# 2. Value Normalization (Price per Square Meter)
# This allows fair comparison between different property sizes
df_spatial['price_per_m2'] = df_spatial['price'] / df_spatial['area']

# 3. Amenity Scoring (Internal Quality)
# Combines booleans into a single score from 0 to 3
# (Higher score = more ready-to-move-in)
df_spatial['quality_score'] = df_spatial['kitchen'] + df_spatial['ac'] + df_spatial['furnished']

# 4. Human-Readable Categories
# Mapping the IDs to names for the Streamlit sidebar filters
category_map = {
    1: 'Apartment (Rental)', 2: 'Land (Sell)', 3: 'Villa (Sell)',
    4: 'Floor (Rental)', 5: 'Villa (Rental)', 6: 'Apartment (Sell)',
    7: 'Building (Sell)', 8: 'Store (Rental)', 9: 'House (Sell)',
    10: 'Esterahah (Sell)', 11: 'House (Rental)', 12: 'Farm (Sell)',
    13: 'Esterahah (Rental)', 14: 'Office (Rental)', 15: 'Land (Rental)',
    16: 'Building (Rental)', 17: 'Warehouse (Rental)', 18: 'Campsite (Rental)',
    19: 'Room (Rental)', 20: 'Store (Sell)', 21: 'Furnished Apartment',
    22: 'Floor (Sell)', 23: 'Chalet (Rental)'
}
df_spatial['category_name'] = df_spatial['category'].map(category_map)

print("Feature Engineering Complete.")
print(f"New Columns added: ['price_per_m2', 'quality_score', 'category_name']")
df_spatial[['price', 'area', 'price_per_m2', 'quality_score', 'category_name']].head()

Feature Engineering Complete.
New Columns added: ['price_per_m2', 'quality_score', 'category_name']


,price,area,price_per_m2,quality_score,category_name
0,1958400.0,816.0,2400.0,0.0,Land (Sell)
1,15078000.0,1077.0,14000.0,0.0,Land (Sell)
2,1050000.0,875.0,1200.0,0.0,Land (Sell)
3,8750000.0,1250.0,7000.0,0.0,Land (Sell)
4,53625000.0,10725.0,5000.0,0.0,Land (Sell)


In [ ]:
cols_to_drop = ['beds', 'livings', 'wc', 'kitchen', 'ac', 'furnished']

df_spatial = df_spatial.drop(columns=cols_to_drop)

print("Columns removed successfully ✅")
print(df_spatial.columns)

Columns removed successfully ✅
Index(['price', 'area', 'category', 'age', 'lat', 'lng', 'user.review',
       'price_per_m2', 'quality_score', 'category_name'],
      dtype='object')


In [ ]:
df_spatial = df_spatial.drop(columns=['quality_score'])


In [ ]:
cols_to_drop1 = ["price_per_m2","category_name"]
df_spatial = df_spatial.drop(columns=cols_to_drop1)

In [14]:
cols_to_drop2 = ["user.review"]
df_spatial = df_spatial.drop(columns=cols_to_drop2)

In [16]:
cols_to_drop3= ["age"]
df_spatial = df_spatial.drop(columns=cols_to_drop3)

In [18]:
df_spatial.head()

,price,area,category,lat,lng
0,1958400.0,816.0,2,24.548930,46.781390
1,15078000.0,1077.0,2,24.754130,46.724820
2,1050000.0,875.0,2,24.560830,46.795360
3,8750000.0,1250.0,2,24.644165,46.659703
4,53625000.0,10725.0,2,24.853706,46.669090


In [19]:

# --- 5. Export Final Dataset ---
output_file = 'Cleaned_Riyadh_Spatial_Real_Estate.csv'

# Save with utf-8-sig to ensure compatibility with Excel and Arabic characters
df_spatial.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"Success! Created '{output_file}' with {len(df_spatial.columns)} features.")

# Download to local machine
files.download(output_file)

Success! Created 'Cleaned_Riyadh_Spatial_Real_Estate.csv' with 5 features.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>